In [2]:
import pandas as pd
import numpy as np

df = pd.read_csv("Used_Car_Price_Prediction.csv")


In [3]:
numeric_features = [
    'yr_mfr',
    'kms_run',
    'total_owners'
]

categorical_features = [
    'fuel_type',
    'city',
    'body_type',
    'transmission',
    'make',
    'model'
]

In [4]:
target = 'sale_price'

In [5]:
df = df[df['sale_price'] > 0].copy()

In [6]:
df['body_type'] = df['body_type'].fillna(df['body_type'].mode()[0])
df['transmission'] = df['transmission'].fillna(df['transmission'].mode()[0])

In [7]:
X = df[numeric_features + categorical_features]
y = df['sale_price']

In [8]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42
)

In [9]:
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder

preprocessor = ColumnTransformer(
    transformers=[
        ('num', 'passthrough', numeric_features),
        (
            'cat',
            OneHotEncoder(
                handle_unknown='ignore',
                sparse_output=False
            ),
            categorical_features
        )
    ]
)

In [10]:
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LinearRegression

basic_model = Pipeline([
    ('preprocessor', preprocessor),
    ('regressor', LinearRegression())
])

basic_model.fit(X_train, y_train)

y_pred = basic_model.predict(X_test)

In [11]:
from sklearn import metrics

mae = metrics.mean_absolute_error(y_test, y_pred)
mse = metrics.mean_squared_error(y_test, y_pred)
rmse = metrics.root_mean_squared_error(y_test, y_pred)
r2 = metrics.r2_score(y_test, y_pred)

print("MAE :", mae)
print("MSE :", mse)
print("RMSE:", rmse)
print("R2  :", r2)

MAE : 59090.95032988885
MSE : 12419436989.674763
RMSE: 111442.52774266546
R2  : 0.8430237692980045


In [12]:
print("Train shape:", X_train.shape)
print("Test shape :", X_test.shape)

print(
    "Encoded train shape:",
    basic_model.named_steps['preprocessor'].transform(X_train).shape
)

print(
    "Encoded test shape:",
    basic_model.named_steps['preprocessor'].transform(X_test).shape
)

Train shape: (5917, 9)
Test shape : (1480, 9)
Encoded train shape: (5917, 234)
Encoded test shape: (1480, 234)


In [13]:
encoder = basic_model.named_steps['preprocessor'].named_transformers_['cat']

for col, categories in zip(categorical_features, encoder.categories_):
    print(col, len(categories))

fuel_type 5
city 13
body_type 5
transmission 2
make 27
model 179


In [14]:
for col in categorical_features:
    train_values = set(X_train[col].unique())
    test_values = set(X_test[col].unique())

    unknown = test_values - train_values

    print(col, "→", unknown)

fuel_type → set()
city → set()
body_type → set()
transmission → set()
make → set()
model → {'linea', 'ml class', 'aria', 'passat', 'indica ev2', 'nuvosport'}


In [15]:
for col in categorical_features:
    train_values = set(X_train[col].unique())
    test_values = set(X_test[col].unique())

    unknown = test_values - train_values

    if unknown:
        print(col, ":", unknown)
        
        for value in unknown:
            print(value, "count:", (X_test[col] == value).sum())

model : {'linea', 'ml class', 'aria', 'passat', 'indica ev2', 'nuvosport'}
linea count: 1
ml class count: 1
aria count: 1
passat count: 1
indica ev2 count: 1
nuvosport count: 1


In [16]:
comparison = pd.DataFrame({
    'actual': y_test,
    'predicted': y_pred
})

comparison['error'] = abs(
    comparison['actual'] - comparison['predicted']
)

print(
    comparison
    .sort_values('error', ascending=False)
    .head(10)
)

       actual     predicted         error
3756  3250000  1.628848e+06  1.621152e+06
2498  2585899  1.545352e+06  1.040547e+06
2861  1917988  8.905933e+05  1.027395e+06
3748  2462277  1.510247e+06  9.520298e+05
2034  1972528  1.106934e+06  8.655942e+05
5492  1830522  1.008512e+06  8.220098e+05
7239   200000  9.935176e+05  7.935176e+05
1210  1538430  9.022556e+05  6.361744e+05
6592   554499  1.138555e+06  5.840558e+05
1565  1566099  1.042282e+06  5.238173e+05


In [17]:
for model in ['aria', 'passat', 'nuvosport', 'linea', 'indica ev2', 'ml class']:
    print("\nMODEL:", model)
    print(
        comparison[
            X_test['model'] == model
        ]
    )


MODEL: aria
      actual      predicted         error
1582  335199  310737.161479  24461.838521

MODEL: passat
      actual      predicted        error
6770  582899  585767.648203  2868.648203

MODEL: nuvosport
      actual      predicted         error
1915  407499  497500.228699  90001.228699

MODEL: linea
      actual      predicted         error
1608  257499  331388.910085  73889.910085

MODEL: indica ev2
      actual      predicted          error
6397  150000  331802.495569  181802.495569

MODEL: ml class
       actual     predicted          error
2034  1972528  1.106934e+06  865594.200568


In [18]:
old_encoded = pd.get_dummies(
    X,
    columns=categorical_features,
    dtype=int
)

print("Old encoded shape:", old_encoded.shape)

Old encoded shape: (7397, 240)


In [19]:
pipeline_encoded_train = basic_model.named_steps[
    'preprocessor'
].transform(X_train)

print("Pipeline encoded shape:", pipeline_encoded_train.shape)

Pipeline encoded shape: (5917, 234)


In [20]:
X_old = pd.get_dummies(
    X,
    columns=categorical_features,
    dtype=int
)

X_old_train = X_old.loc[X_train.index]
X_old_test = X_old.loc[X_test.index]

In [21]:
lr_old = LinearRegression()

lr_old.fit(X_old_train, y_train)

pred_old = lr_old.predict(X_old_test)

print("Old method R2:",
      metrics.r2_score(y_test, pred_old))

Old method R2: 0.8430237692980013


In [22]:
import joblib

joblib.dump(basic_model, "basic_model.pkl")

['basic_model.pkl']